# gpt-oss-20b on vLLM: the text model for turnval

This notebook starts **gpt-oss-20b** in a vLLM server on this machine: an OpenAI-compatible endpoint at `http://<host>:8000/v1`, running in its own process. It then checks that the server answers in JSON through turnval's own client. `turnval_end_to_end.ipynb` uses this server for generating the chats and the summaries: set its `VLLM_URL` to the URL printed in cell 1, and `VLLM_MODEL` to `SERVED_NAME`.

**Needs:**
- an NVIDIA Ampere or newer GPU with about 24 GB free (the model is about 13 GB in MXFP4, plus its KV cache);
- vLLM 0.10.1 or newer;
- the weights, downloaded on the first start or from a local folder.

gpt-oss reasons before it answers. The client asks for `reasoning_effort: "low"` and adds 1,024 thinking tokens to each answer's budget (`TURNVAL_VLLM_REASONING_EFFORT`, `TURNVAL_VLLM_REASONING_TOKENS`).

In [ ]:
# 1. SETTINGS
import os, sys
from pathlib import Path
REPO = str(Path.cwd().resolve().parents[1]) if Path.cwd().name == "notebooks" else str(Path.cwd().resolve())
os.chdir(REPO); sys.path.insert(0, REPO)

MODEL = "openai/gpt-oss-20b"        # the Hugging Face id, or the folder of a local copy of the weights
SERVED_NAME = "openai/gpt-oss-20b"  # the name the server answers to (= VLLM_MODEL in the end-to-end notebook)
HOST = "0.0.0.0"                    # 0.0.0.0: reachable from other machines; 127.0.0.1: this machine only
PORT = 8000
GPU = "0"                           # which GPU(s): "0", or "0,1" with TENSOR_PARALLEL = 2
TENSOR_PARALLEL = 1
GPU_MEMORY_UTILIZATION = 0.85
MAX_MODEL_LEN = 32768
INSTALL_VLLM = False                # True: pip install "vllm>=0.10.1" into this kernel first
# gpt-oss's tokenizer fetches two vocabulary files from the internet on the first request. Without internet, put
# o200k_base.tiktoken and cl100k_base.tiktoken in a folder and name it here.
TIKTOKEN_ENCODINGS_DIR = ""

URL = f"http://localhost:{PORT}/v1"
LOG, PID = os.path.abspath("turnval/out/vllm_gpt_oss.log"), os.path.abspath("turnval/out/vllm_gpt_oss.pid")
os.makedirs("turnval/out", exist_ok=True)
import socket
print("server:", URL, "| from other machines: http://%s:%d/v1" % (socket.getfqdn(), PORT), "| log:", LOG)

In [ ]:
# 2. CHECK THE MACHINE: a GPU, and vLLM new enough for gpt-oss
import subprocess, importlib.util
r = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,memory.used", "--format=csv,noheader"],
                   capture_output=True, text=True)
print(r.stdout.strip() or "no NVIDIA GPU found (nvidia-smi): vLLM needs one for gpt-oss-20b")
if INSTALL_VLLM:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm>=0.10.1"], check=True)
if importlib.util.find_spec("vllm") is None:
    print("vLLM is not installed in this kernel: set INSTALL_VLLM = True above (or pip install 'vllm>=0.10.1')")
else:
    import vllm
    v = tuple(int(x) for x in vllm.__version__.split("+")[0].split(".")[:3] if x.isdigit())
    print("vLLM", vllm.__version__, "- ok" if v >= (0, 10, 1) else "- too old for gpt-oss: install vllm>=0.10.1")

In [ ]:
# 3. START THE SERVER (in its own process; running this cell again reuses a server that is already up)
import json, time, signal, urllib.request, urllib.error

def up(url=URL):
    try:
        with urllib.request.urlopen(url + "/models", timeout=3) as r:
            return [m["id"] for m in json.loads(r.read())["data"]]
    except Exception:
        return None

def tail(n=15):
    try:
        return "".join(open(LOG, encoding="utf-8", errors="replace").readlines()[-n:])
    except OSError:
        return ""

if up():
    print("already serving:", up())
else:
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": GPU}
    if TIKTOKEN_ENCODINGS_DIR:
        env["TIKTOKEN_ENCODINGS_BASE"] = os.path.abspath(TIKTOKEN_ENCODINGS_DIR)
        env["TIKTOKEN_RS_CACHE_DIR"] = os.path.abspath(TIKTOKEN_ENCODINGS_DIR)
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL,
           "--served-model-name", SERVED_NAME, "--port", str(PORT), "--host", HOST,
           "--tensor-parallel-size", str(TENSOR_PARALLEL), "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
           "--max-model-len", str(MAX_MODEL_LEN)]
    proc = subprocess.Popen(cmd, stdout=open(LOG, "w"), stderr=subprocess.STDOUT, env=env, start_new_session=True)
    open(PID, "w").write(str(proc.pid))
    print("starting:", " ".join(cmd))
    t0, last = time.time(), 0
    while not up():
        if proc.poll() is not None:
            raise RuntimeError("the vLLM server stopped:\n" + tail(40))
        if time.time() - t0 > 30 * 60:
            raise TimeoutError("not up after 30 minutes:\n" + tail(40))
        if time.time() - last > 60:
            last = time.time()
            print(f"... {int(time.time() - t0)} s\n" + tail(3))
        time.sleep(5)
    print(f"up after {int(time.time() - t0)} s, serving:", up())

In [ ]:
# 4. CHECK IT ANSWERS IN JSON THROUGH TURNVAL'S CLIENT (what generation will see)
import re
from turnval.local_llm import LocalChat
from turnval.llm import _messages, parse_json_object

def raw(messages, **extra):
    body = {"model": SERVED_NAME, "messages": messages, "max_tokens": 200, **extra}
    req = urllib.request.Request(URL + "/chat/completions", data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=300) as r:
            return "ok: " + str(json.loads(r.read())["choices"][0]["message"].get("content"))[:120]
    except urllib.error.HTTPError as e:
        return f"HTTP {e.code}: {e.read().decode('utf-8', 'replace')[:300]}"

chat = LocalChat(SERVED_NAME, temperature=0.0, max_tokens=300, url=URL)
t0 = time.time()
try:
    reply = chat.invoke(_messages('Reply with ONE JSON object: {"utterance": "<what the caller says>"}',
                                  "INTENT: say yes, you are calling about a fraud alert."))
except Exception as e:                                  # noqa: BLE001
    print("FAILED:", e)
    print("plain request:", raw([{"role": "user", "content": "Say OK."}]))
    log = tail(400)
    if re.search(r"vocab file|HarmonyError|tiktoken|openaipublic", log + str(e)):
        print("LIKELY CAUSE: the tokenizer could not download its vocabulary: set TIKTOKEN_ENCODINGS_DIR (cell 1), "
              "stop (cell 5) and start (cell 3) the server again.")
    elif re.search(r"out of memory|CUDA error|OOM", log):
        print("LIKELY CAUSE: out of GPU memory: lower MAX_MODEL_LEN or GPU_MEMORY_UTILIZATION and restart.")
    else:
        print("see the log:", LOG)
else:
    print(f"{time.time() - t0:.1f} s, {reply.usage_metadata}")
    print(reply.content)
    print("parsed:", parse_json_object(reply.content) or "NOT JSON: raise TURNVAL_VLLM_REASONING_TOKENS or check the log")

In [ ]:
# 5. STOP THE SERVER
try:
    os.killpg(int(open(PID).read()), signal.SIGTERM)
    print("vLLM server stopped")
except (OSError, ValueError) as e:
    print("no server to stop:", e)